In [ ]:
# ================================================================
# Setup: install & imports
# ================================================================
# !pip uninstall -y openai
# !pip install --upgrade --force-reinstall "openai>=1.51.0,<2"
# !pip -q install --upgrade numpy pandas matplotlib pillow scipy tqdm pyarrow

import openai
print("openai version:", openai.__version__)

from openai import OpenAI

import os
import math
import re
import io
import base64
import colorsys
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import display


openai version: 2.9.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ================================================================
# API key and client
# ================================================================
# REPLACE THIS WITH YOUR KEY OR USE COLAB SECRETS.
# In real use, don't hard-code it.
import os
os.environ["OPENAI_API_KEY"] = ""
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Multimodal model name (must support image input)
MODEL = "gpt-4o"  # adjust if you want a different model


In [ ]:
# ============================================================
# Experiment D: Multilingual Full-Grid Comparison (Clean Pipeline)
# ============================================================

# This notebook:
#  - Builds a shared color stimulus grid (from a reference language, e.g. English)
#  - Computes human naming distributions per bin per language
#  - Queries GPT-4o (vision) on the same RGB patches in multiple languages
#  - Maps GPT outputs into human vocab (surface + head)
#  - Computes Jensen–Shannon divergence, in-vocab coverage, confusion matrices
#  - Saves all intermediate results with a timestamped run ID
#  - Emits LaTeX snippets for tables and figures
#
# Requirements:
#  - UW full-grid "cleaned_full_data*.csv" per language in BASE_UW
#  - OPENAI_API_KEY environment variable set
#  - Colab / Python environment with: pandas, numpy, matplotlib, pillow, openai, tqdm

# ----------------------------
# 1. Imports & basic config
# ----------------------------
import os
import re
import json
import math
import unicodedata
from datetime import datetime
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

from pandas.errors import EmptyDataError

from PIL import Image
from io import BytesIO
import base64
import matplotlib.pyplot as plt

from openai import OpenAI

# ----------------------------
# 1.1 Paths & run ID
# ----------------------------

# Root for UW data (adjust to your Drive layout)
BASE_UW = Path("/content/drive/MyDrive/2025_2026/color/uw-data/extracted_langs")

# Output root
OUT_ROOT = Path("/content/drive/MyDrive/2025_2026/color/expD_clean")

# Timestamped run directory to make everything reproducible
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = OUT_ROOT / f"expD_run_{RUN_ID}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run ID: {RUN_ID}")
print(f"Output directory: {OUT_DIR}")

# ----------------------------
# 1.2 Languages & mapping
# ----------------------------

# Languages we want to process (folder names in BASE_UW)
LANG_FOLDERS = [
    "chinese",
    "english",
    "french",
    "german",
    "korean",
    "polish",
    "portuguese",
    "russian",
    "spanish",
]

# Language → code used in prompt
LANG_CODE = {
    "chinese":    "Chinese",
    "english":    "English",
    "french":     "French",
    "german":     "German",
    "korean":     "Korean",
    "polish":     "Polish",
    "portuguese": "Portuguese",
    "russian":    "Russian",
    "spanish":    "Spanish",
}

# Reference language for constructing shared grid / RGB patches
REF_LANG = "english"

# ----------------------------
# 1.3 OpenAI client
# ----------------------------
client = OpenAI()  # assumes OPENAI_API_KEY is set


# ============================================================
# 2. Data loading helpers (UW full-grid)
# ============================================================

def load_cleaned_full(lang_path: Path) -> pd.DataFrame | None:
    """
    Return a non-empty cleaned_full_data*.csv DataFrame or None.
    """
    for fname in os.listdir(lang_path):
        if "cleaned_full_data" in fname and fname.endswith(".csv"):
            fpath = lang_path / fname
            try:
                df = pd.read_csv(fpath)
            except EmptyDataError:
                print(f"  [skip] {fpath.name}: empty file.")
                return None
            except Exception as e:
                print(f"  [skip] {fpath.name}: read error ({e}).")
                return None

            if df.empty:
                print(f"  [skip] {fpath.name}: no data rows.")
                return None

            return df

    return None  # no file found


def detect_color_col(df: pd.DataFrame) -> str:
    """
    Column holding the human color term.
    """
    for c in [
        "standardized_entered_name",
        "standardized_entered",
        "entered_name",
        "name",
    ]:
        if c in df.columns:
            return c

    obj_cols = [c for c in df.columns if df[c].dtype == "object"]
    if obj_cols:
        print("  [warn] Falling back to first object column as color col:", obj_cols[0])
        return obj_cols[0]

    raise ValueError(f"No color-name column in df.columns={list(df.columns)}")


def detect_lab_cols(df: pd.DataFrame) -> tuple[str, str, str]:
    """Return (L, A, B) column names for LAB coordinates."""
    for Lc, Ac, Bc in [
        ("lab_l", "lab_a", "lab_b"),
        ("L", "A", "B"),
        ("l", "a", "b"),
    ]:
        if {Lc, Ac, Bc}.issubset(df.columns):
            return Lc, Ac, Bc
    raise ValueError(f"No LAB columns found in df.columns={list(df.columns)}")


def detect_rgb_cols(df: pd.DataFrame) -> tuple[str | None, str | None, str | None]:
    """Return (r, g, b) column names if present, else (None, None, None)."""
    for Rc, Gc, Bc in [
        ("r", "g", "b"),
        ("R", "G", "B"),
        ("red", "green", "blue"),
    ]:
        if {Rc, Gc, Bc}.issubset(df.columns):
            return Rc, Gc, Bc
    return None, None, None


# ============================================================
# 3. Normalization / label helpers
# ============================================================

def strip_accents(s: str) -> str:
    """
    Remove diacritics; marrón → marron.
    Also fold Russian й→и / Й→И; ß→ss.
    """
    if not isinstance(s, str):
        s = str(s)

    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")

    s = s.replace("й", "и").replace("Й", "И")
    s = s.replace("ß", "ss")
    return s


def normalize_basic(s: str) -> str:
    """Lowercase, strip whitespace/punctuation, remove accents."""
    s = str(s).strip().lower()
    s = strip_accents(s)
    s = re.sub(r'^[\s\.\,\!\?\:\;\-]+', '', s)
    s = re.sub(r'[\s\.\,\!\?\:\;\-]+$', '', s)
    return s


def surface_label(label: str) -> str:
    """
    Surface-level normalization: accent stripping, simple punctuation trim.
    Language-agnostic by default; we’ll treat modifiers via head-label.
    """
    return normalize_basic(label)


# Russian base mapping (simple stems)
RU_BASES = {
    "син": "синий",
    "голуб": "голубой",
    "бирюз": "бирюзовый",
    "зел": "зелёный",
    "зелен": "зелёный",
    "красн": "красный",
    "оранж": "оранжевый",
    "желт": "жёлтый",
    "жёлт": "жёлтый",
    "роз": "розовый",
    "фиол": "фиолетовый",
    "лилов": "фиолетовый",
    "сер": "серый",
    "коричнев": "коричневый",
    "черн": "чёрный",
    "бел": "белый",
}

# Chinese base characters for simple head collapsing
CH_BASE_CHARS = list("红綠绿蓝藍黄黑白灰紫粉橙青棕褐")


def collapse_russian_head(head: str) -> str:
    for stem, canon in RU_BASES.items():
        if stem in head:
            return canon
    return head


# Seed modifiers per language (minimal, you can expand later)
LANG_GENERIC_MODIFIERS_SEED = {
    "english":    ["light", "dark", "pale", "bright", "deep", "medium", "very", "neon"],
    "spanish":    ["claro", "oscuro", "muy", "pálido", "palido"],
    "portuguese": ["claro", "escuro", "muito", "pálido", "palido"],
    "french":     ["clair", "foncé", "fonce", "très", "tres", "pâle", "pale"],
    "german":     ["hell", "dunkel", "sehr"],
    "italian":    ["chiaro", "scuro", "molto"],
    "polish":     ["jasny", "ciemny"],
    "russian":    ["светло", "темно", "ярко", "бледно", "грязно"],
    "korean":     [],
    "chinese":    [],
}

# This will be filled later after we see vocab
LANG_GENERIC_MODIFIERS = {}


def derive_modifiers_from_vocab(
    lang: str,
    surface_vocab: set[str],
    forbidden_heads: set[str] | None = None,
    max_modifiers: int = 10,
) -> list[str]:
    """
    Simple heuristic to auto-derive generic modifiers for this language:
      - Candidates are first/last tokens in multi-word labels
      - Not in forbidden_heads
      - Occur at least freq_min times and co-occur with >= min_heads distinct other tokens
    """
    lang_key = lang.lower()

    if lang_key in {"chinese", "zh", "zhongwen"}:
        return []

    if len(surface_vocab) < 30:
        return []

    if forbidden_heads is None:
        forbidden_heads = set()

    freq_min = 5
    min_heads = 3
    token_stats: dict[str, dict[str, object]] = {}

    for label in surface_vocab:
        toks = label.split()
        if len(toks) < 2:
            continue

        heads = set(toks)
        candidates = {toks[0], toks[-1]}

        for t in candidates:
            if t in forbidden_heads:
                continue
            if t not in token_stats:
                token_stats[t] = {"count": 0, "heads": set()}
            token_stats[t]["count"] += 1
            token_stats[t]["heads"].update(h for h in heads if h != t)

    cand_mods: list[tuple[str, int, int]] = []
    for t, stat in token_stats.items():
        count = stat["count"]
        distinct_heads = len(stat["heads"])
        if count < freq_min or distinct_heads < min_heads:
            continue
        if len(t) <= 2 and lang_key in {"french", "spanish", "portuguese"}:
            continue
        cand_mods.append((t, count, distinct_heads))

    cand_mods.sort(key=lambda x: (-x[1], -x[2]))
    return [t for (t, _, _) in cand_mods[:max_modifiers]]


def head_label(lang: str, label: str) -> str:
    """
    Head-level normalization:
      - lowercase & accent stripping
      - Chinese: strip 色, collapse to base color characters when possible
      - Other langs: remove generic modifiers at edges, keep core token(s)
      - Russian: additional stem collapsing into canonical basic colors
    """
    lang_key = lang.lower()
    base = normalize_basic(label)
    if not base:
        return base

    # Chinese: character-level
    if lang_key in {"chinese", "zh", "zhongwen"}:
        if base.endswith("色"):
            base = base[:-1]
        for ch in CH_BASE_CHARS:
            if ch in base:
                return ch
        return base

    # Korean: strip 색 at end
    if lang_key in {"korean", "ko", "hangul"} and base.endswith("색"):
        base = base[:-1]

    # Remove punctuation
    stripped = re.sub(r"[^\w\s]", " ", base).strip()
    tokens = stripped.split()

    # German: hellblau, dunkelblau
    if lang_key in {"german", "de", "deutsch"} and len(tokens) == 1:
        t = tokens[0]
        for pref in ["hell", "dunkel", "sehr"]:
            if t.startswith(pref) and len(t) > len(pref) + 1:
                return t[len(pref):]

    modifiers = LANG_GENERIC_MODIFIERS.get(lang_key, [])

    if len(tokens) > 1 and modifiers:
        while tokens and tokens[0] in modifiers:
            tokens = tokens[1:]
        while tokens and tokens[-1] in modifiers:
            tokens = tokens[:-1]

    if not tokens:
        head = stripped
    elif len(tokens) == 1:
        head = tokens[0]
    else:
        head = " ".join(tokens)

    if lang_key in {"russian", "ru", "русский"}:
        head = collapse_russian_head(head)

    return head


def map_to_surface_vocab(raw_label: str, vocab: set[str]) -> str:
    surf = surface_label(raw_label)
    if surf in vocab:
        return surf
    if "gray" in vocab and surf == "grey":
        return "gray"
    return "<OTHER_SURF>"


def map_to_head_vocab(lang: str, raw_label: str, vocab: set[str]) -> str:
    head = head_label(lang, raw_label)

    if head in vocab:
        return head
    if "gray" in vocab and head == "grey":
        return "gray"

    tokens = head.split()
    if len(tokens) > 1:
        for cand in (tokens[-1], tokens[0]):
            if cand in vocab:
                return cand

    return "<OTHER_HEAD>"


# ============================================================
# 4. Build shared stimulus grid (from REF_LANG)
# ============================================================

def build_bins_from_df(df: pd.DataFrame, lang: str):
    """
    From full-grid table, produce:
      - bins_surface: { (binL,binA,binB) → {surface_label: count} }
      - rgbs:         { (binL,binA,binB) → (R,G,B) } (if RGB present)
      - labs:         { (binL,binA,binB) → (L,a,b) }
    """
    df = df.copy()
    Lc, Ac, Bc = detect_lab_cols(df)
    Rc, Gc, Bc_rgb = detect_rgb_cols(df)
    color_col = detect_color_col(df)

    df["binL"] = (df[Lc] / 10).round().astype(int)
    df["binA"] = (df[Ac] / 10).round().astype(int)
    df["binB"] = (df[Bc] / 10).round().astype(int)

    bins_surface = {}
    rgbs = {}
    labs = {}

    for _, row in df.iterrows():
        key = (int(row.binL), int(row.binA), int(row.binB))
        raw_label = str(row[color_col])

        surf = surface_label(raw_label)

        if key not in bins_surface:
            bins_surface[key] = {}

        bins_surface[key][surf] = bins_surface[key].get(surf, 0) + 1

        # LAB (we always have them)
        if key not in labs:
            labs[key] = (float(row[Lc]), float(row[Ac]), float(row[Bc]))

        # RGB (optional)
        if key not in rgbs and Rc is not None:
            rgbs[key] = (int(row[Rc]), int(row[Gc]), int(row[Bc_rgb]))

    return bins_surface, labs, rgbs


print("\n=== Building shared stimulus grid from reference language ===")
ref_path = BASE_UW / REF_LANG
ref_df = load_cleaned_full(ref_path)
if ref_df is None:
    raise RuntimeError(f"Reference language {REF_LANG} has no usable full-grid file.")

ref_bins_surface, ref_labs, ref_rgbs = build_bins_from_df(ref_df, REF_LANG)

# Fallback: if RGB missing in ref, we could synthesize from LAB, but that's messy.
missing_rgb_bins = [b for b in ref_labs.keys() if b not in ref_rgbs]
if missing_rgb_bins:
    print(f"[warn] {len(missing_rgb_bins)} reference bins lack RGB; they will be dropped.")
    for b in missing_rgb_bins:
        ref_labs.pop(b, None)
        ref_bins_surface.pop(b, None)

# Build master stimuli DataFrame
stimuli_rows = []
for i, (b, rgb) in enumerate(ref_rgbs.items()):
    (L, a, lab_b) = ref_labs[b]
    stimuli_rows.append({
        "global_bin_id": i,
        "binL": b[0],
        "binA": b[1],
        "binB": b[2],
        "L": L,
        "A": a,
        "B": lab_b,
        "R": rgb[0],
        "G": rgb[1],
        "B_rgb": rgb[2],
    })

stimuli_df = pd.DataFrame(stimuli_rows)
stimuli_df.to_csv(OUT_DIR / "stimuli_master.csv", index=False)
print(f"Saved master stimuli grid with {len(stimuli_df)} bins.")


# ============================================================
# 5. Human distributions per language (over shared grid)
# ============================================================

# ============================================================
# 5. Human distributions per language (over shared grid)
# ============================================================

# Global storage for human distributions / metadata
HUMAN_DISTS = {}  # lang -> dict[bin -> (surface_dist, head_dist, mass)]

# Default thresholds
GLOBAL_MIN_COUNT = 8
GLOBAL_MIN_PROP = 0.20
MAX_BINS_PER_LANG = 150  # soft cap per language

# Per-language overrides (tune as needed)
LANG_FILTER_OVERRIDES = {
    # Looser thresholds for small / sparse languages
    "polish": {
        "min_count": 4,
        "min_prop": 0.15,
        "max_bins": 50,
    },
    "portuguese": {
        "min_count": 4,
        "min_prop": 0.15,
        "max_bins": 50,
    },
    "russian": {
        "min_count": 4,
        "min_prop": 0.15,
        "max_bins": 80,
    },
    # Others use the global defaults unless you add entries here.
}

# Extra base heads that should NOT be treated as modifiers
EXTRA_BASE_HEADS = {
    "english": ["red", "blue", "green", "yellow", "orange", "purple",
                "pink", "brown", "gray", "grey", "black", "white",
                "cyan", "teal", "turquoise", "magenta"],
    "spanish": ["rojo", "azul", "verde", "amarillo", "morado", "lila",
                "rosa", "gris", "negro", "blanco", "cian", "magenta"],
    "french":  ["rouge", "bleu", "vert", "jaune", "violet", "rose",
                "gris", "noir", "blanc", "turquoise"],
    "german":  ["rot", "blau", "grun", "gruen", "gelb", "lila", "violett",
                "rosa", "braun", "grau", "schwarz", "weiss", "weiß"],
    "portuguese": ["vermelho", "azul", "verde", "amarelo", "roxo",
                   "rosa", "cinza", "preto", "branco"],
    "polish": ["czerwony", "niebieski", "zielony", "zolty", "żółty",
               "fioletowy", "rozowy", "różowy", "brazowy", "brązowy",
               "szary", "czarny", "bialy", "biały"],
    "russian": ["синий", "голубой", "бирюзовый", "зелёный", "зеленый",
                "красный", "оранжевый", "жёлтый", "желтый", "розовый",
                "фиолетовый", "серый", "коричневый", "чёрный", "черный",
                "белый"],
    "korean":  [],
    "chinese": [],
}


def build_human_distributions_for_language(lang: str):
    """
    For a given language:
      - load UW full-grid
      - compute bins & human surface vocab
      - infer generic modifiers and recompute head labels
      - restrict to shared stimuli bins (from reference)
      - filter bins by min human mass & dominant proportion
      - keep up to max_bins_per_lang bins (highest human mass)
    """
    print(f"\n=== Human distributions for language: {lang} ===")
    lang_key = lang.lower()
    lang_path = BASE_UW / lang
    df = load_cleaned_full(lang_path)
    if df is None:
        print("  No usable full-grid file; skipping.")
        return

    # First pass: bins + surface only
    bins_surface, labs, _ = build_bins_from_df(df, lang)

    # Limit to bins present in reference stimuli
    ref_bin_keys = set(zip(stimuli_df["binL"], stimuli_df["binA"], stimuli_df["binB"]))
    bins_surface = {b: lbls for b, lbls in bins_surface.items() if b in ref_bin_keys}

    if not bins_surface:
        print("  No overlapping bins with reference grid; skipping.")
        return

    # Surface vocab
    surface_vocab_all = set()
    for _, lbls in bins_surface.items():
        surface_vocab_all.update(lbls.keys())

    # forbidden heads = single tokens + EXTRA_BASE_HEADS
    forbidden_heads = {tok for tok in surface_vocab_all if " " not in tok}
    forbidden_heads.update(EXTRA_BASE_HEADS.get(lang_key, []))

    # Modifiers: seed + auto-derived
    seed_mods = LANG_GENERIC_MODIFIERS_SEED.get(lang_key, [])
    auto_mods = derive_modifiers_from_vocab(lang, surface_vocab_all, forbidden_heads)
    mods = sorted(set(seed_mods) | set(auto_mods))
    LANG_GENERIC_MODIFIERS[lang_key] = mods

    if mods:
        print(f"  [lexicon] Modifiers for {lang}: {mods[:10]} ...")
    else:
        print(f"  [lexicon] No modifiers inferred for {lang}.")

    # Second pass: recompute head-label distributions
    df = df.copy()
    Lc, Ac, Bc = detect_lab_cols(df)
    color_col = detect_color_col(df)
    df["binL"] = (df[Lc] / 10).round().astype(int)
    df["binA"] = (df[Ac] / 10).round().astype(int)
    df["binB"] = (df[Bc] / 10).round().astype(int)

    bins_head = {}
    for _, row in df.iterrows():
        b = (int(row.binL), int(row.binA), int(row.binB))
        if b not in ref_bin_keys:
            continue
        raw_label = str(row[color_col])
        head = head_label(lang, raw_label)
        if b not in bins_head:
            bins_head[b] = {}
        bins_head[b][head] = bins_head[b].get(head, 0) + 1

    # Human mass per bin (surface counts)
    human_mass = {b: sum(lbls.values()) for b, lbls in bins_surface.items()}

    # Distributions
    human_surface_dist = {
        b: {k: v / human_mass[b] for k, v in lbls.items()}
        for b, lbls in bins_surface.items()
    }
    human_head_dist = {}
    for b in bins_surface.keys():
        head_counts = bins_head.get(b, {})
        total = human_mass[b]
        if total > 0 and head_counts:
            human_head_dist[b] = {k: v / total for k, v in head_counts.items()}
        else:
            human_head_dist[b] = {}

    # Per-language filter params
    override = LANG_FILTER_OVERRIDES.get(lang_key, {})
    min_count = override.get("min_count", GLOBAL_MIN_COUNT)
    min_prop = override.get("min_prop", GLOBAL_MIN_PROP)
    max_bins = override.get("max_bins", MAX_BINS_PER_LANG)

    # Filter bins
    good_bins = []
    for b, dist in human_head_dist.items():
        total = human_mass[b]
        if total < min_count:
            continue
        if not dist:
            continue
        top_prop = max(dist.values())
        if top_prop < min_prop:
            continue
        good_bins.append((b, total))

    if not good_bins:
        print(
            f"  No bins survive human filters for {lang} "
            f"(min_count={min_count}, min_prop={min_prop}); skipping."
        )
        return

    # Sort bins by human mass and cap
    good_bins.sort(key=lambda x: -x[1])
    if len(good_bins) > max_bins:
        good_bins = good_bins[:max_bins]

    selected_bins = [b for (b, _) in good_bins]
    print(
        f"  Total overlapping bins: {len(bins_surface)}, "
        f"retained after filters: {len(selected_bins)} "
        f"(min_count={min_count}, min_prop={min_prop})"
    )

    # Save per-language human summary
    rows = []
    for b in selected_bins:
        surf_dist = human_surface_dist[b]
        head_dist = human_head_dist[b]
        rows.append({
            "lang": lang,
            "binL": b[0],
            "binA": b[1],
            "binB": b[2],
            "human_mass": human_mass[b],
            "dominant_prop": max(head_dist.values()) if head_dist else 0.0,
            "n_surface_labels": len(surf_dist),
            "n_head_labels": len(head_dist),
        })
    human_summary_df = pd.DataFrame(rows)
    human_summary_df.to_csv(OUT_DIR / f"human_bins_{lang}.csv", index=False)

    # Build final dict: bin -> (surface_dist, head_dist, mass)
    dist_dict = {}
    for b in selected_bins:
        dist_dict[b] = (human_surface_dist[b], human_head_dist[b], human_mass[b])
    HUMAN_DISTS[lang] = dist_dict


for lang in LANG_FOLDERS:
    build_human_distributions_for_language(lang)

# Drop languages with no usable bins
LANGS_ACTIVE = [lang for lang in LANG_FOLDERS if lang in HUMAN_DISTS]
print("\nActive languages in Experiment D:", LANGS_ACTIVE)



# ============================================================
# 6. LLM query: GPT-4o + image patches
# ============================================================

def rgb_to_data_url(rgb, size: int = 64) -> str:
    """Make a solid RGB swatch PNG and return as data: URL."""
    r, g, b = rgb
    img = Image.new("RGB", (size, size), (int(r), int(g), int(b)))
    buf = BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/png;base64,{b64}"


def query_llm_color_name(rgb, lang_code: str, temperature: float = 0.7) -> str:
    """
    Query GPT-4o with an RGB patch and return raw text.
    """
    img_url = rgb_to_data_url(rgb)
    prompt = (
        f"Name this color in {lang_code}. "
        f"Answer with exactly one natural color word or very short phrase in {lang_code}, "
        f"no English, no translation, no explanation."
    )

    resp = client.responses.create(
        model="gpt-4o",
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": img_url},
            ],
        }],
        max_output_tokens=30,
        temperature=temperature,
    )

    try:
        text = resp.output_text
    except AttributeError:
        # fallback for older clients
        pieces = []
        for item in getattr(resp, "output", []):
            for c in getattr(item, "content", []):
                if getattr(c, "type", None) == "output_text":
                    pieces.append(c.text)
        text = " ".join(pieces) if pieces else ""

    return text.strip()


def jensen_shannon(p: dict[str, float], q: dict[str, float]) -> float:
    keys = set(p.keys()) | set(q.keys())
    pvec = np.array([p.get(k, 0.0) for k in keys], dtype=float)
    qvec = np.array([q.get(k, 0.0) for k in keys], dtype=float)
    m = 0.5 * (pvec + qvec)

    def kl(a, b):
        mask = a > 0
        return np.sum(a[mask] * np.log2(a[mask] / b[mask]))

    return 0.5 * kl(pvec, m) + 0.5 * kl(qvec, m)


# ============================================================
# 7. Run Experiment D for each language
# ============================================================

SAMPLES_PER_BIN = 3      # set to 5 or 10 if you want more stability
TEMPERATURE = 0.7        # moderate

all_results = []
RAW_RESP_DIR = OUT_DIR / "raw_llm_outputs"
RAW_RESP_DIR.mkdir(exist_ok=True)

for lang in LANGS_ACTIVE:
    print(f"\n=== Running Experiment D for language: {lang} ===")
    dist_dict = HUMAN_DISTS[lang]
    lang_code = LANG_CODE[lang]
    lang_key = lang.lower()

    # Build mapping from bin -> RGB
    # Join via binL/binA/binB on stimuli_df
    lang_rows = []
    for b in dist_dict.keys():
        sel = stimuli_df[
            (stimuli_df["binL"] == b[0]) &
            (stimuli_df["binA"] == b[1]) &
            (stimuli_df["binB"] == b[2])
        ]
        if sel.empty:
            continue
        row = sel.iloc[0]
        lang_rows.append({
            "bin": b,
            "R": int(row["R"]),
            "G": int(row["G"]),
            "B_rgb": int(row["B_rgb"]),
        })

    if not lang_rows:
        print("  No bins have RGB patches; skipping LLM queries.")
        continue

    # Turn into dict
    RGB_BY_BIN = {r["bin"]: (r["R"], r["G"], r["B_rgb"]) for r in lang_rows}
    active_bins = list(RGB_BY_BIN.keys())
    print(f"  LLM will be queried on {len(active_bins)} bins for {lang}.")

    # Container for raw LLM outputs
    raw_records = []

    for b in tqdm(active_bins, desc=f"{lang} bins"):
        rgb = RGB_BY_BIN[b]

        samples = []
        for k in range(SAMPLES_PER_BIN):
            txt = query_llm_color_name(rgb, lang_code, temperature=TEMPERATURE)
            samples.append(txt)
            raw_records.append({
                "lang": lang,
                "binL": b[0],
                "binA": b[1],
                "binB": b[2],
                "sample_idx": k,
                "rgb_R": rgb[0],
                "rgb_G": rgb[1],
                "rgb_B": rgb[2],
                "gpt_raw": txt,
            })

        # We'll compute metrics below after we have full list of raw labels
        # for this bin across languages.

    # Save raw LLM outputs for this language (for offline re-analysis)
    raw_df = pd.DataFrame(raw_records)
    raw_df.to_csv(RAW_RESP_DIR / f"llm_raw_{lang}.csv", index=False)
    print(f"  Saved raw LLM outputs for {lang} ({len(raw_df)} rows).")

    # -------------------------
    # Build distributions + metrics
    # -------------------------
    # Rebuild by bin to align with human distributions
    lang_results = []
    human_info = dist_dict  # b -> (surf_dist, head_dist, human_mass)

    for b in active_bins:
        if b not in human_info:
            continue
        surf_human, head_human, human_mass = human_info[b]

        # Collect LLM labels for this bin
        subset = raw_df[
            (raw_df["binL"] == b[0]) &
            (raw_df["binA"] == b[1]) &
            (raw_df["binB"] == b[2])
        ]
        raw_labels = list(subset["gpt_raw"])

        # Surface-level mapping
        surface_vocab = set(surf_human.keys())
        llm_surface_bucketed = []
        n_surface_in_vocab = 0
        for lbl in raw_labels:
            mapped = map_to_surface_vocab(lbl, surface_vocab)
            llm_surface_bucketed.append(mapped)
            if mapped != "<OTHER_SURF>":
                n_surface_in_vocab += 1

        llm_surface_counts = Counter(llm_surface_bucketed)
        total_s = sum(llm_surface_counts.values())
        llm_surface_dist = {k: v / total_s for k, v in llm_surface_counts.items()}

        # Extend human with OTHER bucket if needed
        human_surface_ext = dict(surf_human)
        if "<OTHER_SURF>" not in human_surface_ext:
            human_surface_ext["<OTHER_SURF>"] = 0.0

        js_surface = jensen_shannon(human_surface_ext, llm_surface_dist)
        surface_in_vocab_frac = n_surface_in_vocab / SAMPLES_PER_BIN

        human_surface_top = max(surf_human.items(), key=lambda kv: kv[1])[0]
        llm_surface_top = max(llm_surface_dist.items(), key=lambda kv: kv[1])[0]

        # Head-level mapping
        head_vocab = set(head_human.keys())
        llm_head_bucketed = []
        n_head_in_vocab = 0
        for lbl in raw_labels:
            mapped = map_to_head_vocab(lang, lbl, head_vocab)
            llm_head_bucketed.append(mapped)
            if mapped != "<OTHER_HEAD>":
                n_head_in_vocab += 1

        llm_head_counts = Counter(llm_head_bucketed)
        total_h = sum(llm_head_counts.values())
        llm_head_dist = {k: v / total_h for k, v in llm_head_counts.items()}

        human_head_ext = dict(head_human)
        if "<OTHER_HEAD>" not in human_head_ext:
            human_head_ext["<OTHER_HEAD>"] = 0.0

        js_head = jensen_shannon(human_head_ext, llm_head_dist)
        head_in_vocab_frac = n_head_in_vocab / SAMPLES_PER_BIN

        human_head_top = max(head_human.items(), key=lambda kv: kv[1])[0]
        llm_head_top = max(llm_head_dist.items(), key=lambda kv: kv[1])[0]

        lang_results.append({
            "lang": lang,
            "binL": b[0],
            "binA": b[1],
            "binB": b[2],
            "js_surface": js_surface,
            "llm_surface_in_vocab_frac": surface_in_vocab_frac,
            "js_head": js_head,
            "llm_head_in_vocab_frac": head_in_vocab_frac,
            "human_surface_top": human_surface_top,
            "llm_surface_top": llm_surface_top,
            "human_head_top": human_head_top,
            "llm_head_top": llm_head_top,
            "human_labels_surface": len(surf_human),
            "human_labels_head": len(head_human),
            "samples": SAMPLES_PER_BIN,
            "human_mass": human_mass,
            "dominant_prop_head": max(head_human.values()) if head_human else 0.0,
        })

    lang_res_df = pd.DataFrame(lang_results)
    lang_res_df.to_csv(OUT_DIR / f"metrics_{lang}.csv", index=False)
    all_results.append(lang_res_df)

# Combine all languages
if all_results:
    all_results_df = pd.concat(all_results, ignore_index=True)
    all_results_df.to_csv(OUT_DIR / "metrics_all_languages.csv", index=False)
    print("\nSaved metrics for all languages.")
else:
    print("\nNo results to aggregate.")


# ============================================================
# 8. Confusion matrices (head-level, weighted by human mass)
# ============================================================

CONF_DIR = OUT_DIR / "confusions"
CONF_DIR.mkdir(exist_ok=True)

head_confusions = {}

if all_results:
    for lang, df_lang in all_results_df.groupby("lang"):
        mat = pd.crosstab(
            df_lang["human_head_top"],
            df_lang["llm_head_top"],
            values=df_lang["human_mass"],
            aggfunc="sum",
        ).fillna(0).astype(int)

        head_confusions[lang] = mat
        mat.to_csv(CONF_DIR / f"confusion_head_{lang}.csv")

        print(f"\nHead-level confusion matrix for {lang}:")
        print(mat)


# ============================================================
# 9. Patch images for appendix
# ============================================================

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

def save_stimuli_patch_grid(stimuli_df: pd.DataFrame, out_path: Path,
                            n_cols: int = 24, patch_size: int = 16):
    """
    Save a grid image of all stimuli patches in LAB-bin order.
    Mainly for appendix illustration; not meant to be perceptually perfect.
    """
    n = len(stimuli_df)
    n_rows = math.ceil(n / n_cols)

    # create blank canvas
    img = Image.new("RGB", (n_cols * patch_size, n_rows * patch_size), (255, 255, 255))

    for idx, row in stimuli_df.reset_index(drop=True).iterrows():
        r, g, b = int(row["R"]), int(row["G"]), int(row["B_rgb"])
        patch = Image.new("RGB", (patch_size, patch_size), (r, g, b))
        row_i = idx // n_cols
        col_j = idx % n_cols
        img.paste(patch, (col_j * patch_size, row_i * patch_size))

    img.save(out_path)
    print(f"Saved patch grid to {out_path}")


save_stimuli_patch_grid(stimuli_df, FIG_DIR / "expD_fullgrid_patches.png",
                        n_cols=24, patch_size=16)


# ============================================================
# 10. LaTeX auto-generation
# ============================================================

LATEX_PATH = OUT_DIR / "expD_latex_snippets.tex"

latex_lines = []

if all_results:
    # Summary table: per-language mean JS / in-vocab frac (surface & head)
    summary = (
        all_results_df
        .groupby("lang")[["js_surface", "llm_surface_in_vocab_frac",
                          "js_head", "llm_head_in_vocab_frac"]]
        .mean()
        .reset_index()
    )

    # Pretty mapping for language names
    LANG_DISPLAY = {
        "chinese": "Chinese",
        "english": "English",
        "french": "French",
        "german": "German",
        "korean": "Korean",
        "polish": "Polish",
        "portuguese": "Portuguese",
        "russian": "Russian",
        "spanish": "Spanish",
    }

    latex_lines.append("% Auto-generated Experiment D summary table\n")
    latex_lines.append("\\begin{table}[t]\n")
    latex_lines.append("\\centering\n")
    latex_lines.append("\\small\n")
    latex_lines.append("\\begin{tabular}{lcccc}\n")
    latex_lines.append("\\hline\n")
    latex_lines.append("Language & JS$_{\\text{surf}}$ & In-vocab$_{\\text{surf}}$ & JS$_{\\text{head}}$ & In-vocab$_{\\text{head}}$ \\\\\n")
    latex_lines.append("\\hline\n")

    for _, row in summary.iterrows():
        lang = row["lang"]
        name = LANG_DISPLAY.get(lang, lang)
        js_surf = row["js_surface"]
        in_surf = row["llm_surface_in_vocab_frac"]
        js_head = row["js_head"]
        in_head = row["llm_head_in_vocab_frac"]
        latex_lines.append(
            f"{name} & {js_surf:.3f} & {in_surf:.2f} & {js_head:.3f} & {in_head:.2f} \\\\\n"
        )

    latex_lines.append("\\hline\n")
    latex_lines.append("\\end{tabular}\n")
    latex_lines.append("\\caption{Experiment~D: Mean Jensen--Shannon divergence and in-vocabulary fraction for human vs. GPT-4o color naming across languages. Surface-level uses full human strings; head-level collapses modifiers and near-synonymous heads.}\n")
    latex_lines.append("\\label{tab:expD-summary}\n")
    latex_lines.append("\\end{table}\n\n")

# Figure snippet for patch grid
latex_lines.append("% Patch grid for Experiment D stimuli\n")
latex_lines.append("\\begin{figure}[t]\n")
latex_lines.append("\\centering\n")
latex_lines.append("\\includegraphics[width=0.55\\linewidth]{figures/expD_fullgrid_patches.png}\n")
latex_lines.append("\\caption{Full-grid color stimuli used in Experiment~D (reference language grid). Each square is one LAB bin rendered as an RGB swatch; the same patches are shown to GPT-4o across languages.}\n")
latex_lines.append("\\label{fig:expD-patches}\n")
latex_lines.append("\\end{figure}\n")

with open(LATEX_PATH, "w", encoding="utf-8") as f:
    f.writelines(latex_lines)

print(f"\nWrote LaTeX snippets to {LATEX_PATH}")
print("Done.")


Run ID: 20251209_205309
Output directory: /content/drive/MyDrive/2025_2026/color/expD_clean/expD_run_20251209_205309

=== Building shared stimulus grid from reference language ===
Saved master stimuli grid with 1267 bins.

=== Human distributions for language: chinese ===
  [lexicon] No modifiers inferred for chinese.
  Total overlapping bins: 1108, retained after filters: 150 (min_count=8, min_prop=0.2)

=== Human distributions for language: english ===
  [lexicon] Modifiers for english: ['baby', 'bright', 'dark', 'darker', 'deep', 'dirty', 'faded', 'hot', 'light', 'medium'] ...
  Total overlapping bins: 1267, retained after filters: 150 (min_count=8, min_prop=0.2)

=== Human distributions for language: french ===
  [lexicon] Modifiers for french: ['clair', 'fluo', 'fonce', 'foncé', 'green', 'moyen', 'pale', 'pâle', 'sale', 'sombre'] ...
  Total overlapping bins: 1000, retained after filters: 27 (min_count=8, min_prop=0.2)

=== Human distributions for language: german ===
  [lexicon] 

chinese bins: 100%|██████████| 150/150 [07:39<00:00,  3.07s/it]


  Saved raw LLM outputs for chinese (450 rows).

=== Running Experiment D for language: english ===
  LLM will be queried on 150 bins for english.


english bins: 100%|██████████| 150/150 [06:46<00:00,  2.71s/it]


  Saved raw LLM outputs for english (450 rows).

=== Running Experiment D for language: french ===
  LLM will be queried on 27 bins for french.


...

  Saved raw LLM outputs for portuguese (150 rows).

=== Running Experiment D for language: russian ===
  LLM will be queried on 80 bins for russian.


russian bins: 100%|██████████| 80/80 [03:37<00:00,  2.72s/it]


  Saved raw LLM outputs for russian (240 rows).

=== Running Experiment D for language: spanish ===
  LLM will be queried on 92 bins for spanish.


spanish bins: 100%|██████████| 92/92 [04:36<00:00,  3.00s/it]


  Saved raw LLM outputs for spanish (276 rows).

Saved metrics for all languages.

Head-level confusion matrix for chinese:
llm_head_top    <OTHER_HEAD>   棕   橙  灰   粉    紫   红    绿    蓝   青   黄
human_head_top                                                        
棕                          0  10   0  0   0    0   0    0    0   0   0
橙                          0   0  11  0   0    0   0    0    0   0   0
灰                          0   0   0  9   0    0   0    0    0   0   0
粉                         19   0   0  0  20    9   0    0    0   0   0
紫                          0   0   0  0  27  193   0    0    0   0   0
綠                          0   0   0  0   0    0   0    0    0  11   0
红                         13   0  21  0  30    0  98    0    0   0   0
绿                         22   0   0  0   0    0   0  440    0  65   0
蓝                          0   0   0  0   0   40   0    0  366  66   0
藍                          0   0   0  0   0    0   0    0    9   0   0
青                       

In [ ]:
# ============================================================
# How many human subjects per language (UW full-grid)
# ============================================================

def detect_subject_col(df: pd.DataFrame) -> str | None:
    """
    Try to locate a subject/worker ID column in the UW full-grid files.
    Returns the column name or None if nothing sensible is found.
    """
    candidate_cols = [
        "subject_id", "participant_id", "workerid", "worker_id",
        "user_id", "user", "participant", "subject"
    ]
    for c in candidate_cols:
        if c in df.columns:
            return c

    # If nothing obvious, try any column that looks like an ID:
    for c in df.columns:
        if "id" in c.lower() or "worker" in c.lower() or "subject" in c.lower():
            return c

    return None


def count_subjects_per_language(base_dir: Path, lang_folders: list[str]):
    """
    For each language folder, load cleaned_full_data*.csv,
    detect subject column, and count unique subjects.
    """
    summary_rows = []

    for lang in lang_folders:
        lang_path = base_dir / lang
        df = load_cleaned_full(lang_path)
        if df is None:
            print(f"[subjects] {lang}: no usable full-grid file.")
            continue

        subj_col = detect_subject_col(df)
        if subj_col is None:
            print(f"[subjects] {lang}: no subject ID column found in columns={list(df.columns)}")
            n_subj = None
        else:
            n_subj = df[subj_col].nunique()
            print(f"[subjects] {lang}: {n_subj} unique subjects (column='{subj_col}')")

        summary_rows.append({"lang": lang, "n_subjects": n_subj, "subject_col": subj_col})

    return pd.DataFrame(summary_rows)


subjects_df = count_subjects_per_language(BASE_UW, LANG_FOLDERS)
subjects_df.to_csv(OUT_DIR / "human_subject_counts_per_language.csv", index=False)
subjects_df


[subjects] chinese: 5686 unique subjects (column='colorNameId')
[subjects] english: 145709 unique subjects (column='colorNameId')
[subjects] french: 2968 unique subjects (column='colorNameId')
[subjects] german: 3721 unique subjects (column='colorNameId')
[subjects] korean: 13507 unique subjects (column='colorNameId')
[subjects] polish: 1085 unique subjects (column='colorNameId')
[subjects] portuguese: 1661 unique subjects (column='colorNameId')
[subjects] russian: 1682 unique subjects (column='colorNameId')
[subjects] spanish: 4124 unique subjects (column='colorNameId')


,lang,n_subjects,subject_col
0,chinese,5686,colorNameId
1,english,145709,colorNameId
2,french,2968,colorNameId
3,german,3721,colorNameId
4,korean,13507,colorNameId
5,polish,1085,colorNameId
6,portuguese,1661,colorNameId
7,russian,1682,colorNameId
8,spanish,4124,colorNameId


In [ ]:
import pandas as pd

# ---------------------------------------------------------
# Load your aggregated Experiment D metrics
# (modify this to your actual file)
# ---------------------------------------------------------
df = pd.read_csv("expD_all_lang_metrics.csv")

# Expected columns:
#   lang       – language name (string)
#   bin_id     – unique bin
#   js_head    – JS divergence between human vs LLM head categories

# ---------------------------------------------------------
# Define thresholds (match LaTeX table)
# ---------------------------------------------------------
GOOD_MAX = 0.3
BAD_MIN  = 0.7

def classify_js(js):
    if js <= GOOD_MAX:
        return "good"
    elif js >= BAD_MIN:
        return "bad"
    else:
        return "mid"

df["quality"] = df["js_head"].apply(classify_js)

# ---------------------------------------------------------
# Compute summary table
# ---------------------------------------------------------
summary = (
    df.groupby("lang")["quality"]
      .value_counts()
      .unstack(fill_value=0)
      .rename_axis(None, axis=1)
      .reset_index()
)

# Ensure consistent ordering
for col in ["good", "mid", "bad"]:
    if col not in summary.columns:
        summary[col] = 0

# Add number of bins
bin_counts = df.groupby("lang")["bin_id"].nunique().rename("#Bins")
summary = summary.merge(bin_counts, on="lang")

# Reorder columns
summary = summary[["lang", "#Bins", "good", "mid", "bad"]]

print(summary)


In [ ]:
from pathlib import Path
import pandas as pd

# ---- Adjust RUN_ID if you rerun Experiment D ----
RUN_ID = "20251209_205309"

OUT_ROOT = Path("/content/drive/MyDrive/2025_2026/color/expD_clean")
OUT_DIR  = OUT_ROOT / f"expD_run_{RUN_ID}"

METRICS_CSV = OUT_DIR / "metrics_all_languages.csv"
df = pd.read_csv(METRICS_CSV)

df.head()


,lang,binL,binA,binB,js_surface,llm_surface_in_vocab_frac,js_head,llm_head_in_vocab_frac,human_surface_top,llm_surface_top,human_head_top,llm_head_top,human_labels_surface,human_labels_head,samples,human_mass,dominant_prop_head
0,chinese,7,-2,-4,0.380930,1.0,0.190875,1.0,天蓝色,天蓝色,蓝,蓝,11,3,3,24,0.666667
1,chinese,8,-4,-2,0.562783,1.0,0.562783,1.0,蓝色,青色,蓝,青,10,4,3,21,0.619048
2,chinese,4,3,-8,0.377070,1.0,0.146015,1.0,蓝色,蓝色,蓝,蓝,12,2,3,19,0.736842
3,chinese,8,-7,7,0.405937,1.0,0.268666,1.0,绿色,绿色,绿,绿,10,3,3,18,0.555556
4,chinese,6,8,-5,1.000000,0.0,0.836124,1.0,紫色,<OTHER_SURF>,紫,粉,13,4,3,17,0.470588


In [ ]:
GOOD_MAX = 0.3
BAD_MIN  = 0.7

def classify_js(js):
    if js <= GOOD_MAX:
        return "good"
    elif js >= BAD_MIN:
        return "bad"
    else:
        return "mid"

df["quality"] = df["js_head"].apply(classify_js)


In [ ]:
# unique bins per lang (just to be safe, in case of duplicates)
df["bin_key"] = list(zip(df["binL"], df["binA"], df["binB"]))
# If you know each (lang, binL,binA,binB) is already unique, you can skip this.

# Count good/mid/bad
summary = (
    df.groupby("lang")["quality"]
      .value_counts()
      .unstack(fill_value=0)
      .rename_axis(None, axis=1)
      .reset_index()
)

for col in ["good", "mid", "bad"]:
    if col not in summary.columns:
        summary[col] = 0

# Number of bins per language
bin_counts = df.groupby("lang")["bin_key"].nunique().rename("#Bins")
summary = summary.merge(bin_counts, on="lang")

# Reorder columns
summary = summary[["lang", "#Bins", "good", "mid", "bad"]]

summary


,lang,#Bins,good,mid,bad
0,chinese,150,107,28,15
1,english,150,71,55,24
2,french,27,9,13,5
3,german,48,24,15,9
4,korean,150,42,57,51
5,polish,36,11,14,11
6,portuguese,50,21,21,8
7,russian,80,25,37,18
8,spanish,92,33,32,27


In [ ]:
LANG_LABEL = {
    "chinese":    "Chinese",
    "english":    "English",
    "french":     "French",
    "german":     "German",
    "korean":     "Korean",
    "polish":     "Polish",
    "portuguese": "Portuguese",
    "russian":    "Russian",
    "spanish":    "Spanish",
}

def to_latex(summary_df):
    rows = []
    for _, r in summary_df.iterrows():
        lang_disp = LANG_LABEL.get(r["lang"], r["lang"].capitalize())
        rows.append(
            f"{lang_disp} & {int(r['#Bins'])} & {int(r['good'])} & "
            f"{int(r['mid'])} & {int(r['bad'])} \\\\"
        )
    body = "\n".join(rows)
    latex = f"""
\\begin{{table}}[H]
\\centering
\\small
\\begin{{tabular}}{{lrrrr}}
\\hline
Lang & \\#Bins & Good & Mid & Bad \\\\
\\hline
{body}
\\hline
\\end{{tabular}}
\\caption{{Experiment~D: counts of bins labeled
\\textit{{good}}, \\textit{{mid}}, or \\textit{{bad}}
based on JS$_\\text{{head}}$ thresholds (good $\\le 0.3$, bad $\\ge 0.7$).}}
\\label{{tab:expD-bin-quality}}
\\end{{table}}
"""
    return latex

print(to_latex(summary))



\begin{table}[H]
\centering
\small
\begin{tabular}{lrrrr}
\hline
Lang & \#Bins & Good & Mid & Bad \\
\hline
Chinese & 150 & 107 & 28 & 15 \\
English & 150 & 71 & 55 & 24 \\
French & 27 & 9 & 13 & 5 \\
German & 48 & 24 & 15 & 9 \\
Korean & 150 & 42 & 57 & 51 \\
Polish & 36 & 11 & 14 & 11 \\
Portuguese & 50 & 21 & 21 & 8 \\
Russian & 80 & 25 & 37 & 18 \\
Spanish & 92 & 33 & 32 & 27 \\
\hline
\end{tabular}
\caption{Experiment~D: counts of bins labeled
\textit{good}, \textit{mid}, or \textit{bad}
based on JS$_\text{head}$ thresholds (good $\le 0.3$, bad $\ge 0.7$).}
\label{tab:expD-bin-quality}
\end{table}



In [ ]:
from pathlib import Path
import pandas as pd

RUN_ID = "20251209_205309"  # adjust if you rerun

OUT_ROOT = Path("/content/drive/MyDrive/2025_2026/color/expD_clean")
OUT_DIR  = OUT_ROOT / f"expD_run_{RUN_ID}"

metrics_path = OUT_DIR / "metrics_all_languages.csv"
df = pd.read_csv(metrics_path)

df.head()


,lang,binL,binA,binB,js_surface,llm_surface_in_vocab_frac,js_head,llm_head_in_vocab_frac,human_surface_top,llm_surface_top,human_head_top,llm_head_top,human_labels_surface,human_labels_head,samples,human_mass,dominant_prop_head
0,chinese,7,-2,-4,0.380930,1.0,0.190875,1.0,天蓝色,天蓝色,蓝,蓝,11,3,3,24,0.666667
1,chinese,8,-4,-2,0.562783,1.0,0.562783,1.0,蓝色,青色,蓝,青,10,4,3,21,0.619048
2,chinese,4,3,-8,0.377070,1.0,0.146015,1.0,蓝色,蓝色,蓝,蓝,12,2,3,19,0.736842
3,chinese,8,-7,7,0.405937,1.0,0.268666,1.0,绿色,绿色,绿,绿,10,3,3,18,0.555556
4,chinese,6,8,-5,1.000000,0.0,0.836124,1.0,紫色,<OTHER_SURF>,紫,粉,13,4,3,17,0.470588


In [ ]:
# Unique bin key per row
df["bin_key"] = list(zip(df["binL"], df["binA"], df["binB"]))

# Per-language basic stats
grouped = df.groupby("lang")

bins_per_lang = grouped["bin_key"].nunique().rename("Bins")
mean_js        = grouped["js_surface"].mean().rename("Mean_JS")
mean_js_head   = grouped["js_head"].mean().rename("Mean_JS_head")
in_vocab_surf  = grouped["llm_surface_in_vocab_frac"].mean().rename("In_vocab_surf")
in_vocab_head  = grouped["llm_head_in_vocab_frac"].mean().rename("In_vocab_head")
human_mass     = grouped["human_mass"].sum().rename("Human_mass")

summary = pd.concat(
    [bins_per_lang, mean_js, mean_js_head,
     in_vocab_surf, in_vocab_head, human_mass],
    axis=1
).reset_index()

summary


,lang,Bins,Mean_JS,Mean_JS_head,In_vocab_surf,In_vocab_head,Human_mass
0,chinese,150,0.600822,0.250525,0.760000,0.951111,1612
1,english,150,0.475931,0.413542,0.993333,0.993333,37480
2,french,27,0.600723,0.467269,0.716049,0.814815,253
3,german,48,0.454440,0.379201,0.847222,0.868056,456
4,korean,150,0.579449,0.558473,0.717778,0.722222,3654
5,polish,36,0.550991,0.530598,0.601852,0.620370,154
6,portuguese,50,0.579154,0.414885,0.606667,0.846667,249
7,russian,80,0.593371,0.458983,0.629167,0.725000,369
8,spanish,92,0.560577,0.468627,0.724638,0.811594,844


In [ ]:
LANG_LABEL = {
    "chinese":    "Chinese",
    "english":    "English",
    "french":     "French",
    "german":     "German",
    "korean":     "Korean",
    "polish":     "Polish",
    "portuguese": "Portuguese",
    "russian":    "Russian",
    "spanish":    "Spanish",
}

def make_expD_summary_table(summary_df: pd.DataFrame) -> str:
    rows = []
    for _, r in summary_df.iterrows():
        lang = r["lang"]
        name = LANG_LABEL.get(lang, lang.capitalize())
        bins_ = int(r["Bins"])
        mean_js = r["Mean_JS"]
        mean_js_head = r["Mean_JS_head"]
        inv_s = r["In_vocab_surf"]
        inv_h = r["In_vocab_head"]
        mass = int(round(r["Human_mass"]))

        rows.append(
            f"{name} & {bins_:3d} & {mean_js:.3f} & {mean_js_head:.3f} & "
            f"{inv_s:.3f} & {inv_h:.3f} & {mass} \\\\"
        )

    body = "\n".join(rows)

    latex = f"""
\\begin{{table*}}[t]
\\centering
\\small
\\begin{{tabular}}{{lrrrrrr}}
\\hline
Language & Bins & Mean JS & Mean JS$_\\text{{head}}$ &
In-vocab (surf) & In-vocab (head) & Human mass \\\\
\\hline
{body}
\\hline
\\end{{tabular}}
\\caption{{Experiment~D: Per-language alignment between GPT-4o and human color naming on full CIELAB grids.}}
\\label{{tab:D-all-lang-summary}}
\\end{{table*}}
"""
    return latex

print(make_expD_summary_table(summary))



\begin{table*}[t]
\centering
\small
\begin{tabular}{lrrrrrr}
\hline
Language & Bins & Mean JS & Mean JS$_\text{head}$ &
In-vocab (surf) & In-vocab (head) & Human mass \\
\hline
Chinese & 150 & 0.601 & 0.251 & 0.760 & 0.951 & 1612 \\
English & 150 & 0.476 & 0.414 & 0.993 & 0.993 & 37480 \\
French &  27 & 0.601 & 0.467 & 0.716 & 0.815 & 253 \\
German &  48 & 0.454 & 0.379 & 0.847 & 0.868 & 456 \\
Korean & 150 & 0.579 & 0.558 & 0.718 & 0.722 & 3654 \\
Polish &  36 & 0.551 & 0.531 & 0.602 & 0.620 & 154 \\
Portuguese &  50 & 0.579 & 0.415 & 0.607 & 0.847 & 249 \\
Russian &  80 & 0.593 & 0.459 & 0.629 & 0.725 & 369 \\
Spanish &  92 & 0.561 & 0.469 & 0.725 & 0.812 & 844 \\
\hline
\end{tabular}
\caption{Experiment~D: Per-language alignment between GPT-4o and human color naming on full CIELAB grids.}
\label{tab:D-all-lang-summary}
\end{table*}

